# JARVIS + free tool belt on Kaggle

Ye notebook isi repo ka backend chalta hai, `cloudflared` quick tunnel ke through ek public
URL deta hai, aur JARVIS ko Claude (ya local Ollama) se jodta hai.

**Pehle ye karo**

1. Notebook settings → **Internet: On** (Claude aur tunnel dono ke liye zaroori).
2. Add-ons → **Secrets** → `ANTHROPIC_API_KEY` aur `BACKEND_API_KEY` add karo.
3. Cell 1 chalao, phir Cell 2. Neeche se public URL copy karo.

Bina `ANTHROPIC_API_KEY` ke bhi backend chalta hai — JARVIS rules mode me jawab dega aur
saaf bolega ki model nahi laga.


In [ ]:
!git clone --depth 1 https://github.com/zeeburgers00-glitch/universal-ai-backend.git /kaggle/working/uai
%cd /kaggle/working/uai
!pip install -q -r requirements.txt


In [ ]:
# Backend + tunnel. Ctrl-C / stop cell se dono band ho jaate hain.
# Provider order badalna ho: --provider ollama  (pehle `ollama serve` chalao)
!python deploy/kaggle/jarvis_kaggle.py --port 8000 --provider anthropic


In [ ]:
# Doosre cell se check karo (public URL upar se paste karo)
import json, os, urllib.request

BASE = "https://REPLACE-ME.trycloudflare.com"   # <- apna URL
KEY  = os.environ.get("BACKEND_API_KEY") or "kaggle-local-key"

def call(path, payload=None):
    data = json.dumps(payload).encode() if payload else None
    req = urllib.request.Request(f"{BASE}{path}", data=data,
                                 headers={"X-API-Key": KEY, "Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=120).read())

print(call("/v1/belt/status")["belt"]["ready"], "tools ready")
print(call("/v1/jarvis/chat", {"message": "kitne tools hain"})["reply"])
print(call("/v1/jarvis/chat", {"message": "tool git op=log limit=3"})["reply"][:400])
